In [3]:
from dotenv import load_dotenv
import plotly.io as pio
import neptune

load_dotenv('.tokens.env', override=True)

project = neptune.init_project(project='mtyrol/drop')
project

[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/mtyrol/drop/


In [15]:
df = project.fetch_runs_table(tag=['inner_eval']).to_pandas()

In [36]:
# parameters/subgoal_generator/subgoal_generation_kwargs/num_return_sequences
ALL_BUDGET_VALUES = [1, 2, 5, 10, 25, 40, 50, 60, 70, 80, 90, 100, 120, 140, 160, 180, 200, 250, 300, 350, 400, 450, 500, 600, 800, 1000]


from dataclasses import dataclass
from itertools import product
from typing import Collection
import plotly.express as px
import plotly.graph_objects as go

import pandas as pd

@dataclass
class NeptuneParam:
    full_name_on_neptune: str
    short_name: str
    
params = list(map(
    lambda x: NeptuneParam(
        full_name_on_neptune=x,
        short_name=x.split('/')[-1]
    ),
    [
        'parameters/subgoal_generator/subgoal_generation_kwargs/num_return_sequences',
        'parameters/subgoal_generator/generator_k_list',
        *[f'solved/rate/{n}_nodes' for n in ALL_BUDGET_VALUES]
    ]
))

from abc import ABC, abstractmethod

class PlotFbn(ABC):
    @abstractmethod
    def __call__(self,
                 runs_table: pd.DataFrame,
                 grouping_params: list[NeptuneParam],
                 values_params: list[NeptuneParam]) -> go.Figure:
        pass
    

class PlotSuccessRateByBudget(PlotFbn):
    def __call__(self,
                 runs_table: pd.DataFrame,
                 grouping_params: list[NeptuneParam],
                 values_params: list[NeptuneParam]) -> go.Figure:
        grouping_col_names = [param.full_name_on_neptune for param in grouping_params]
        unique_values = {
            param.short_name: runs_table[param.full_name_on_neptune].unique()
            for param in grouping_params
        }
        
        all_combinations = runs_table[grouping_col_names].drop_duplicates().apply(
            lambda row: tuple(row),
            axis=1
        ).tolist()
        assert len(all_combinations) == len(set(all_combinations)), "There should be no duplicate combinations of grouping parameters"
        all_combinations = set(all_combinations)
        
        filtered_runs = runs_table[
            runs_table[grouping_col_names].apply(
                lambda row: tuple(row) in all_combinations,
                axis=1
            )
        ]
        
        # assert only one run per combination
        assert len(filtered_runs) == len(all_combinations), "There should be one run per combination of grouping parameters"
        filtered_runs = filtered_runs[grouping_col_names + [param.full_name_on_neptune for param in values_params]]
        
        # rename to short names
        filtered_runs = filtered_runs.rename(
            columns={param.full_name_on_neptune: param.short_name for param in grouping_params + values_params}
        )
        
        fig = go.Figure()
        
        for group in all_combinations:
            group_filter = (filtered_runs[grouping_params[0].short_name] == group[0]) & \
                           (filtered_runs[grouping_params[1].short_name] == group[1])
            group_data = filtered_runs[group_filter]
            
            success_rate_list = [group_data[param.short_name].values[0] for param in values_params]
            fig.add_trace(
                go.Scatter(
                    x=ALL_BUDGET_VALUES,
                    y=success_rate_list,
                    mode='lines+markers',
                    name=f"{grouping_params[0].short_name}={group[0]}, {grouping_params[1].short_name}={group[1]}"
                )
            )
        fig.update_layout(
            title='Success Rate by Budget',
            xaxis_title='Budget (number of nodes)',
            yaxis_title='Success Rate',
            legend_title='Group'
        )
        
        fig.update_xaxes(tickvals=ALL_BUDGET_VALUES, ticktext=[str(b) for b in ALL_BUDGET_VALUES])
        fig.update_yaxes(tickformat=".0%")
        pio.templates.default = "plotly_white"
        fig.update_layout(template=pio.templates.default)
        fig.update_layout(width=1000, height=600)
        fig.update_layout(margin=dict(l=20, r=20, t=50, b=20))
        
        return fig

PlotSuccessRateByBudget()(runs_table=df,
                          grouping_params=params[:2],
                          values_params=params[2:])

[NeptuneParam(full_name_on_neptune='parameters/subgoal_generator/subgoal_generation_kwargs/num_return_sequences', short_name='num_return_sequences'),
 NeptuneParam(full_name_on_neptune='parameters/subgoal_generator/generator_k_list', short_name='generator_k_list'),
 NeptuneParam(full_name_on_neptune='solved/rate/1_nodes', short_name='1_nodes'),
 NeptuneParam(full_name_on_neptune='solved/rate/2_nodes', short_name='2_nodes'),
 NeptuneParam(full_name_on_neptune='solved/rate/5_nodes', short_name='5_nodes'),
 NeptuneParam(full_name_on_neptune='solved/rate/10_nodes', short_name='10_nodes'),
 NeptuneParam(full_name_on_neptune='solved/rate/25_nodes', short_name='25_nodes'),
 NeptuneParam(full_name_on_neptune='solved/rate/40_nodes', short_name='40_nodes'),
 NeptuneParam(full_name_on_neptune='solved/rate/50_nodes', short_name='50_nodes'),
 NeptuneParam(full_name_on_neptune='solved/rate/60_nodes', short_name='60_nodes'),
 NeptuneParam(full_name_on_neptune='solved/rate/70_nodes', short_name='70_no

In [23]:
params

['parameters/subgoal_generator/subgoal_generation_kwargs/num_return_sequences',
 'parameters/subgoal_generator/generator_k_list',
 'solved/rate/1_nodes',
 'solved/rate/2_nodes',
 'solved/rate/5_nodes',
 'solved/rate/10_nodes',
 'solved/rate/25_nodes',
 'solved/rate/40_nodes',
 'solved/rate/50_nodes',
 'solved/rate/60_nodes',
 'solved/rate/70_nodes',
 'solved/rate/80_nodes',
 'solved/rate/90_nodes',
 'solved/rate/100_nodes',
 'solved/rate/120_nodes',
 'solved/rate/140_nodes',
 'solved/rate/160_nodes',
 'solved/rate/180_nodes',
 'solved/rate/200_nodes',
 'solved/rate/250_nodes',
 'solved/rate/300_nodes',
 'solved/rate/350_nodes',
 'solved/rate/400_nodes',
 'solved/rate/450_nodes',
 'solved/rate/500_nodes',
 'solved/rate/600_nodes',
 'solved/rate/800_nodes',
 'solved/rate/1000_nodes']

In [24]:
df[params]

,parameters/subgoal_generator/subgoal_generation_kwargs/num_return_sequences,parameters/subgoal_generator/generator_k_list,solved/rate/1_nodes,solved/rate/2_nodes,solved/rate/5_nodes,solved/rate/10_nodes,solved/rate/25_nodes,solved/rate/40_nodes,solved/rate/50_nodes,solved/rate/60_nodes,...,solved/rate/200_nodes,solved/rate/250_nodes,solved/rate/300_nodes,solved/rate/350_nodes,solved/rate/400_nodes,solved/rate/450_nodes,solved/rate/500_nodes,solved/rate/600_nodes,solved/rate/800_nodes,solved/rate/1000_nodes
0,4,[8],0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.006250,...,0.568750,0.668750,0.737500,0.787500,0.787500,0.793750,0.800000,0.806250,0.831250,0.850000
1,3,[8],0.0,0.0,0.0,0.0,0.000000,0.007692,0.034615,0.073077,...,0.711538,0.757692,0.780769,0.800000,0.807692,0.819231,0.823077,0.846154,0.869231,0.884615
2,2,[8],0.0,0.0,0.0,0.0,0.007692,0.069231,0.126923,0.223077,...,0.800000,0.815385,0.838462,0.869231,0.880769,0.884615,0.884615,0.884615,0.888462,0.892308
3,1,[8],0.0,0.0,0.0,0.0,0.103846,0.357692,0.526923,0.603846,...,0.711538,0.711538,0.711538,0.711538,0.711538,0.711538,0.711538,0.711538,0.711538,0.711538
4,1,"[8, 4]",0.0,0.0,0.0,0.0,0.103846,0.361538,0.550000,0.646154,...,0.865385,0.873077,0.876923,0.888462,0.892308,0.892308,0.892308,0.896154,0.896154,0.907692
5,1,"[8, 4, 1]",0.0,0.0,0.0,0.0,0.103846,0.361538,0.550000,0.646154,...,0.869231,0.880769,0.884615,0.900000,0.919231,0.926923,0.926923,0.934615,0.934615,0.946154
